In [1]:
import pandas as pd
import requests

In [2]:
BASE_URL = "https://guest.api.arcadia.pinnacle.com/0.1/leagues/493"

In [3]:
matchups_response = requests.get(f"{BASE_URL}/matchups")
matchups_response.raise_for_status()

with open("matchups.json", "wb") as f:
    f.write(matchups_response.content)

In [4]:
odds_response = requests.get(f"{BASE_URL}/markets/straight")
odds_response.raise_for_status()
with open("odds.json", "wb") as f:
    f.write(odds_response.content)

In [ ]:
# Parse NCAA Tournament Winner participants and odds
pd.set_option("display.max_rows", None)

all_odds = odds_response.json()

for tourney_matchup in matchups_response.json():
    if "Winner" in tourney_matchup.get("special", {}).get("category", ""):
        matchup_id = tourney_matchup["id"]
        if len(tourney_matchup["participants"]) < 3:
            continue
        participants = {p["id"]: p["name"] for p in tourney_matchup["participants"]}

        # Find the odds associated with that matchup ID
        tourney_odds = next((o for o in all_odds if o["matchupId"] == matchup_id), None)
        if tourney_odds:
            data = []
            for p in tourney_odds["prices"]:
                participant_id = p["participantId"]
                price = p["price"]
                team_name = participants.get(
                    participant_id, f"Unknown ({participant_id})"
                )
                # Calculate implied probability from American odds
                if price > 0:
                    implied_prob = 1 / (price / 100 + 1)
                else:
                    implied_prob = 1 / (100 / abs(price) + 1)
                data.append(
                    {"team": team_name, "price": price, "implied_prob": implied_prob}
                )
            df = pd.DataFrame(data)

            # Compute vig-adjusted (fair) implied probability
            total_implied = df["implied_prob"].sum()
            df["vig_adjusted_prob"] = df["implied_prob"] / total_implied

            # Sort by highest win probability
            df = df.sort_values("vig_adjusted_prob", ascending=False).reset_index(
                drop=True
            )
            display(
                df[["team", "vig_adjusted_prob"]]
                .style.format({"vig_adjusted_prob": "{:.2%}"})
                .set_caption(tourney_matchup.get("special", {}).get("category", ""))
            )
        else:
            print("No odds found for the tournament winner matchup.")

,team,vig_adjusted_prob
0,Duke,49.88%
1,Connecticut,12.34%
2,Michigan State,9.33%
3,St. John's,7.45%
4,Kansas,6.43%
5,Louisville,5.30%
6,UCLA,3.06%
7,Ohio State,2.22%
8,TCU,0.99%
9,UCF,0.82%


,team,vig_adjusted_prob
0,Arizona,48.66%
1,Purdue,17.16%
2,Gonzaga,11.87%
3,Arkansas,7.99%
4,Wisconsin,5.28%
5,Miami Florida,1.86%
6,Texas,1.65%
7,BYU,1.55%
8,Utah State,1.18%
9,Villanova,0.95%


,team,vig_adjusted_prob
0,Michigan,46.60%
1,Iowa State,21.28%
2,Virginia,9.20%
3,Alabama,6.36%
4,Tennessee,4.83%
5,Texas Tech,4.00%
6,Kentucky,2.31%
7,Georgia,1.81%
8,Saint Louis,0.82%
9,SMU,0.66%


,team,vig_adjusted_prob
0,Duke,17.39%
1,Michigan,16.36%
2,Arizona,15.93%
3,Florida,8.58%
4,Houston,7.67%
5,Iowa State,4.08%
6,Illinois,3.54%
7,Purdue,3.40%
8,Connecticut,2.42%
9,Arkansas,1.93%


,team,vig_adjusted_prob
0,Florida,30.43%
1,Houston,26.83%
2,Illinois,18.93%
3,Vanderbilt,8.34%
4,Nebraska,5.56%
5,Iowa,2.69%
6,North Carolina,1.63%
7,Saint Marys CA,1.49%
8,Texas A&M,1.09%
9,Clemson,1.03%
